In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import time
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

from src.db_utils import load_sql

print("Модули импортированы")

Модули импортированы


In [2]:
df = load_sql("SELECT * FROM documents")
print(f"Всего отзывов: {len(df)}")

embeddings = np.load('../data/processed/all_review_embeddings.npy')
print(f"Эмбеддинги: {embeddings.shape}")

Всего отзывов: 131669
Эмбеддинги: (131669, 312)


In [3]:
CHROMA_PATH = '../data/chroma_db'

client = chromadb.PersistentClient(path=CHROMA_PATH)

print(f"ChromaDB клиент создан")
print(f"Данные хранятся в: {CHROMA_PATH}")

collections = client.list_collections()
print(f"Существующие коллекции: {[c.name for c in collections]}")

ChromaDB клиент создан
Данные хранятся в: ../data/chroma_db
Существующие коллекции: []


In [8]:
COLLECTION_NAME = 'kinopoisk_reviews'

try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Старая коллекция '{COLLECTION_NAME}' удалена")
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={
        'description': 'Отзывы Кинопоиска с эмбеддингами',
        'hnsw:space': 'cosine'
    }
)

print(f"Коллекция '{COLLECTION_NAME}' создана")
print(f"Размер: {collection.count()} документов")

Коллекция 'kinopoisk_reviews' создана
Размер: 0 документов


In [9]:
BATCH_SIZE = 1000

print(f"Загружаем отзывы в ChromaDB батчами по {BATCH_SIZE}...")
start_time = time.time()

ids = [f"review_{i}" for i in range(len(df))]
documents = df['text'].tolist()
metadatas = [{'category': cat} for cat in df['category']]
embeddings_list = embeddings.tolist()

total_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in range(total_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min((batch_idx + 1) * BATCH_SIZE, len(df))
    
    batch_ids = ids[start_idx:end_idx]
    batch_docs = documents[start_idx:end_idx]
    batch_meta = metadatas[start_idx:end_idx]
    batch_embs = embeddings_list[start_idx:end_idx]
    
    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_meta,
        embeddings=batch_embs
    )
    
    print(f"Батч {batch_idx + 1}/{total_batches} загружен ({len(batch_ids)} документов)")

elapsed = time.time() - start_time
print(f"\nВсего загружено {collection.count()} документов за {elapsed:.2f} секунд")

Загружаем отзывы в ChromaDB батчами по 1000...
Батч 1/132 загружен (1000 документов)
Батч 2/132 загружен (1000 документов)
Батч 3/132 загружен (1000 документов)
Батч 4/132 загружен (1000 документов)
Батч 5/132 загружен (1000 документов)
Батч 6/132 загружен (1000 документов)
Батч 7/132 загружен (1000 документов)
Батч 8/132 загружен (1000 документов)
Батч 9/132 загружен (1000 документов)
Батч 10/132 загружен (1000 документов)
Батч 11/132 загружен (1000 документов)
Батч 12/132 загружен (1000 документов)
Батч 13/132 загружен (1000 документов)
Батч 14/132 загружен (1000 документов)
Батч 15/132 загружен (1000 документов)
Батч 16/132 загружен (1000 документов)
Батч 17/132 загружен (1000 документов)
Батч 18/132 загружен (1000 документов)
Батч 19/132 загружен (1000 документов)
Батч 20/132 загружен (1000 документов)
Батч 21/132 загружен (1000 документов)
Батч 22/132 загружен (1000 документов)
Батч 23/132 загружен (1000 документов)
Батч 24/132 загружен (1000 документов)
Батч 25/132 загружен (1000

In [10]:
peek = collection.peek(limit=3)

print("Первые 3 документа:")
for i in range(len(peek['ids'])):
    print(f"\nID: {peek['ids'][i]}")
    print(f"Категория: {peek['metadatas'][i]['category']}")
    print(f"Текст: {peek['documents'][i][:100]}...")
    print(f"Эмбеддинг (первые 5 чисел): {peek['embeddings'][i][:5]}")

Первые 3 документа:

ID: review_0
Категория: neg
Текст: В 2003-ем году под руководством малоизвестного режиссёра Кларка Джонсона студия 'Columbia Pictures' ...
Эмбеддинг (первые 5 чисел): [-0.02396114  0.02146724 -0.04512804 -0.03257334 -0.05825168]

ID: review_1
Категория: neg
Текст: Грустно и печально. Грустно от того, что довольно неплохой боевичок «Спецназ города ангелов» с Колин...
Эмбеддинг (первые 5 чисел): [ 0.02283395 -0.01638165 -0.03443768 -0.02607204 -0.02889955]

ID: review_2
Категория: neg
Текст: Давным-давно Кира Найтли ворвалась на экран отважной девчонкой, готовой сменить быт на приключения, ...
Эмбеддинг (первые 5 чисел): [ 0.02486033 -0.03120288  0.0215217  -0.03867351 -0.03049918]


In [11]:
embedder = SentenceTransformer('cointegrated/rubert-tiny2')

def chroma_search(query, top_k: int = 5) -> pd.DataFrame:

    query_embedding = embedder.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    
    data = []
    for i in range(len(results['ids'][0])):
        data.append({
            'id': results['ids'][0][i],
            'text': results['documents'][0][i],
            'category': results['metadatas'][0][i]['category'],
            'distance': results['distances'][0][i]
        })
    
    return pd.DataFrame(data)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

query = "кино было великолепным, актёры сыграли превосходно"

chroma_results = chroma_search(query, top_k=5)
chroma_ids = chroma_results['id'].tolist()

query_embedding = embedder.encode([query], convert_to_numpy=True)
similarities = cosine_similarity(query_embedding, embeddings)[0]
top_indices = np.argsort(similarities)[-5:][::-1]
numpy_ids = [f"review_{idx}" for idx in top_indices]

print(f"Запрос: '{query}'\n")
print("ChromaDB IDs:", chroma_ids)
print("Numpy IDs:   ", numpy_ids)
print(f"\nСовпадение: {len(set(chroma_ids) & set(numpy_ids))} из 5")

Запрос: 'кино было великолепным, актёры сыграли превосходно'

ChromaDB IDs: ['review_22503', 'review_101296', 'review_117733', 'review_79736', 'review_72661']
Numpy IDs:    ['review_22503', 'review_101296', 'review_117733', 'review_79736', 'review_72661']

Совпадение: 5 из 5


In [16]:
def chroma_search_filtered(query: str, category: str = None, top_k: int = 5) -> pd.DataFrame:
    query_embedding = embedder.encode([query]).tolist()
    
    where_filter = None
    if category:
        where_filter = {'category': category}
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where=where_filter
    )
    
    data = []
    for i in range(len(results['ids'][0])):
        data.append({
            'id': results['ids'][0][i],
            'text': results['documents'][0][i],
            'category': results['metadatas'][0][i]['category'],
            'similarity': 1 - results['distances'][0][i]
        })
    
    return pd.DataFrame(data)

query = "фильм — настоящий шедевр"
results = chroma_search_filtered(query, category='neg', top_k=5)

print(f"Запрос: '{query}'")
print(f"Фильтр: category = 'neg'\n")
for i, row in results.iterrows():
    print(f"{i+1}. [Сходство: {row['similarity']:.3f}] [{row['category']}]")
    print(f"   {row['text'][:150]}...")
    print()

Запрос: 'фильм — настоящий шедевр'
Фильтр: category = 'neg'

1. [Сходство: 0.670] [neg]
   Дешёвка! Очередная попытка "косить" под Голливуд! Анимационные вставки - единственное, что можно вспомнить из этого фильма. А всё остальное: сюжет, иг...

2. [Сходство: 0.656] [neg]
   Фильм предсказуемый,не страшный и явно рассчитанный на вторую часть! Единственное,за что можно похвалить фильм -  за толковый подбор актеров.

3 из 10...

3. [Сходство: 0.655] [neg]
   Еще год назад данный фильм стали называть русской игрой престолов. Везде писали и говорили, что снимается нечто грандиозное для нашего кино. Пиарили к...

4. [Сходство: 0.655] [neg]
   А это комедия? Действительно, есть смешные моменты, но на комедию, по-моему, не тянет. Вообще фильм скучноват, непонятно, где здесь идея, где сюжет. Н...

5. [Сходство: 0.653] [neg]
   Казалось бы, история Древней Руси - это идеальный плацдарм для создания исторических фильмов жанра 'фэнтези'. Но если в дело вмешивается всеми 'любимы...



In [17]:
query = "отличный фильм с интересным сюжетом"

print("БЕЗ фильтра:")
results_all = chroma_search_filtered(query, top_k=5)
print(results_all['category'].value_counts())
print()

print("Фильтр: category = 'pos':")
results_pos = chroma_search_filtered(query, category='pos', top_k=5)
print(results_pos['category'].value_counts())
print()

print("Фильтр: category = 'neg':")
results_neg = chroma_search_filtered(query, category='neg', top_k=5)
print(results_neg['category'].value_counts())

БЕЗ фильтра:
category
pos    4
neu    1
Name: count, dtype: int64

Фильтр: category = 'pos':
category
pos    5
Name: count, dtype: int64

Фильтр: category = 'neg':
category
neg    5
Name: count, dtype: int64


In [18]:
complex_filter = {
    '$or': [
        {'category': 'pos'},
        {'category': 'neu'}
    ]
}

results = collection.query(
    query_embeddings=embedder.encode(["отличный фильм"]).tolist(),
    n_results=5,
    where=complex_filter
)

print(f"Найдено отзывов с фильтром $or: {len(results['ids'][0])}")
print("Категории:", [m['category'] for m in results['metadatas'][0]])

Найдено отзывов с фильтром $or: 5
Категории: ['pos', 'pos', 'neu', 'neu', 'pos']


In [19]:
import time

test_queries = [
    "отличный фильм",
    "ужасный фильм, зря потратил время",
    "интересный сюжет и хорошая актёрская игра",
    "фильм — настоящий шедевр кинематографа",
    "скучно, не рекомендую"
]

print("Numpy-поиск (линейный):")
numpy_times = []
for query in test_queries:
    start = time.time()
    for _ in range(100):
        q_emb = embedder.encode([query], convert_to_numpy=True)
        sims = cosine_similarity(q_emb, embeddings)[0]
        top_idx = np.argsort(sims)[-5:][::-1]
    elapsed = (time.time() - start) / 100 * 1000  # в мс
    numpy_times.append(elapsed)
    print(f"  '{query[:30]}...': {elapsed:.2f} мс")

print(f"\nСреднее: {np.mean(numpy_times):.2f} мс\n")

print("ChromaDB (ANN):")
chroma_times = []
for query in test_queries:
    start = time.time()
    for _ in range(100):
        q_emb = embedder.encode([query]).tolist()
        results = collection.query(query_embeddings=q_emb, n_results=5)
    elapsed = (time.time() - start) / 100 * 1000  # в мс
    chroma_times.append(elapsed)
    print(f"  '{query[:30]}...': {elapsed:.2f} мс")

print(f"\nСреднее: {np.mean(chroma_times):.2f} мс\n")

# Итоговое сравнение
print("="*50)
print(f"Ускорение: {np.mean(numpy_times) / np.mean(chroma_times):.1f}x")

Numpy-поиск (линейный):
  'отличный фильм...': 192.45 мс
  'ужасный фильм, зря потратил вр...': 198.32 мс
  'интересный сюжет и хорошая акт...': 197.70 мс
  'фильм — настоящий шедевр кинем...': 190.62 мс
  'скучно, не рекомендую...': 190.33 мс

Среднее: 193.89 мс

ChromaDB (ANN):
  'отличный фильм...': 8.22 мс
  'ужасный фильм, зря потратил вр...': 9.00 мс
  'интересный сюжет и хорошая акт...': 8.26 мс
  'фильм — настоящий шедевр кинем...': 8.58 мс
  'скучно, не рекомендую...': 8.19 мс

Среднее: 8.45 мс

Ускорение: 22.9x


In [20]:
del client

client2 = chromadb.PersistentClient(path=CHROMA_PATH)
collection2 = client2.get_collection(COLLECTION_NAME)

print(f"Данные сохранились! Размер коллекции: {collection2.count()}")

results = collection2.query(
    query_embeddings=embedder.encode(["отличный фильм"]).tolist(),
    n_results=3
)
print(f"Поиск работает! Найдено: {len(results['ids'][0])} отзывов")

Данные сохранились! Размер коллекции: 131669
Поиск работает! Найдено: 3 отзывов
